# Hybrid Extraction Test
Verifies that Gemma handles question generation and Claude Haiku handles extraction.
Run from `CAPSTONE2026_SPRING/` root with Ollama running.

In [1]:
import importlib
import intake_engine.llm_adapter
import intake_engine.app_flow
import intake_engine.IntakeState
import intake_engine.llm_extractor
importlib.reload(intake_engine.llm_adapter)
importlib.reload(intake_engine.llm_extractor)
importlib.reload(intake_engine.IntakeState)
importlib.reload(intake_engine.app_flow)
print('Reloaded.')

Reloaded.


## 1. Adapter wiring — confirm two separate backends

In [2]:
from intake_engine.app_flow import IntakeAppFlow
from intake_engine.llm_adapter import OllamaAdapter, ClaudeHaikuAdapter

app = IntakeAppFlow(conn=None)

print('Question generator backend:', type(app.adapter).__name__)
print('Extraction backend:        ', type(app.extraction_adapter).__name__)

assert isinstance(app.adapter, OllamaAdapter), f'Expected OllamaAdapter, got {type(app.adapter).__name__}'
assert isinstance(app.extraction_adapter, ClaudeHaikuAdapter), f'Expected ClaudeHaikuAdapter, got {type(app.extraction_adapter).__name__}'
assert app.extractor.llm_callable == app.extraction_adapter.generate, 'Extractor not using Haiku'

print('\nPASS: Gemma for questions, Haiku for extraction')

Question generator backend: OllamaAdapter
Extraction backend:         ClaudeHaikuAdapter

PASS: Gemma for questions, Haiku for extraction


## 2. Haiku extraction — valid JSON for a yes/no target

In [3]:
import json
from intake_engine.llm_extractor import BoundedLLMExtractor
from intake_engine.llm_adapter import ClaudeHaikuAdapter
from intake_engine.config import API_KEYS
from intake_engine.state.templates import make_empty_intake

haiku = ClaudeHaikuAdapter(api_key=API_KEYS['anthropic'])
extractor = BoundedLLMExtractor(haiku.generate)

intake = make_empty_intake()
intake['chief_complaint_primary'] = 'seizure'

test_cases = [
    ('ask_neurologic_symptoms', 'No weakness or numbness.', False),
    ('ask_head_trauma', 'Yes, I hit my head yesterday.', True),
    ('ask_fever_or_neck_stiffness', 'No fever, no stiff neck.', False),
]

print('=== Haiku extraction — yes/no targets ===')
for intent, answer, expected in test_cases:
    try:
        update = extractor.extract_update(
            intent=intent,
            patient_answer=answer,
            intake_state=intake,
        )
        target = intent.replace('ask_', '', 1)
        value = update.get('set_fields', {}).get(f'policy_answers.{target}')
        status = '\033[92m PASS\033[0m' if value == expected else '\033[91m FAIL\033[0m'
        print(f'  {status}  {intent}("{answer[:30]}...") -> {value} (expected {expected})')
    except Exception as e:
        print(f'  \033[91m FAIL\033[0m  {intent} — ERROR: {e}')

=== Haiku extraction — yes/no targets ===
   PASS  ask_neurologic_symptoms("No weakness or numbness....") -> False (expected False)
   PASS  ask_head_trauma("Yes, I hit my head yesterday....") -> True (expected True)
   PASS  ask_fever_or_neck_stiffness("No fever, no stiff neck....") -> False (expected False)


## 3. Haiku extraction — complex free-text targets

In [4]:
from pprint import pprint

print('=== Haiku extraction — free-text targets ===')

free_text_cases = [
    ('ask_associated_symptoms', 'I have nausea, sensitivity to light, and some neck stiffness.'),
    ('ask_aggravating_factors', 'Bright light makes it much worse, and bending forward too.'),
    ('ask_relieving_factors', 'Lying down in a dark room helps a lot.'),
]

for intent, answer in free_text_cases:
    try:
        update = extractor.extract_update(
            intent=intent,
            patient_answer=answer,
            intake_state=intake,
        )
        print(f'\n  {intent}:')
        print(f'    answer: "{answer}"')
        print(f'    set_fields:    {update.get("set_fields", {})}')
        print(f'    append_fields: {update.get("append_fields", {})}')
    except Exception as e:
        print(f'  FAIL {intent} — ERROR: {e}')

=== Haiku extraction — free-text targets ===

  ask_associated_symptoms:
    answer: "I have nausea, sensitivity to light, and some neck stiffness."
    set_fields:    {}
    append_fields: {'hpi.associated_symptoms': ['nausea', 'sensitivity to light', 'neck stiffness'], 'pertinent_positives': ['nausea', 'sensitivity to light', 'neck stiffness']}

  ask_aggravating_factors:
    answer: "Bright light makes it much worse, and bending forward too."
    set_fields:    {}
    append_fields: {'hpi.aggravating_factors': ['bright light', 'bending forward']}

  ask_relieving_factors:
    answer: "Lying down in a dark room helps a lot."
    set_fields:    {}
    append_fields: {'hpi.relieving_factors': ['lying down in a dark room']}


## 4. No EXTRACTOR FALLBACK in a full intake turn

In [5]:
import sys
from io import StringIO

print('=== Full intake turn — no extractor fallback ===')

app = IntakeAppFlow(conn=None)
app.new_session(session_name='haiku_extraction_test', auto_save=False)
app.state.data['chief_complaint_primary'] = 'i had a seizure'
app.start_intake(auto_save=False)

# Capture stdout to check for EXTRACTOR FALLBACK messages
old_stdout = sys.stdout
sys.stdout = buffer = StringIO()

answers = [
    'No weakness or numbness.',
    'No head injury.',
    'No fever or stiff neck.',
    'Yes, I was confused after.',
    'No, just the one episode.',
    'No, not getting worse.',
    'My wife witnessed it, no warning.',
    'About two hours ago.',
    'About 90 seconds.',
    '7 out of 10.',
]

for answer in answers:
    app.submit_answer(answer, auto_save=False)

sys.stdout = old_stdout
output = buffer.getvalue()

fallbacks = [line for line in output.split('\n') if 'EXTRACTOR FALLBACK' in line]

if fallbacks:
    print(f'  \033[91m FAIL\033[0m  {len(fallbacks)} extractor fallback(s) detected:')
    for f in fallbacks:
        print(f'    {f}')
else:
    print('  \033[92m PASS\033[0m  No extractor fallbacks — Haiku returned valid JSON for all turns')

state = app.get_state()
print(f'\n  HPI so far:')
from pprint import pprint
pprint({k: v for k, v in state['hpi'].items() if v and v != []})
print(f'\n  Policy answers so far:')
pprint({k: v for k, v in state['policy_answers'].items() if v is not None})

=== Full intake turn — no extractor fallback ===
   PASS  No extractor fallbacks — Haiku returned valid JSON for all turns

  HPI so far:
{'duration': 'about 90 seconds',
 'onset': 'About two hours ago',
 'severity': '7/10'}

  Policy answers so far:
{'aura_features': [],
 'confusion_or_ams': True,
 'fever_or_neck_stiffness': False,
 'head_trauma': False,
 'neurologic_symptom_terms': [],
 'neurologic_symptoms': False,
 'prodrome_witnessed_loss_of_consciousness': False,
 'radiation': [],
 'rapid_worsening': False,
 'syncope_or_presyncope': False}


## 5. Question generation still uses Gemma

In [6]:
print('=== Question generation — still using Gemma ===')

app2 = IntakeAppFlow(conn=None)
app2.new_session(session_name='gemma_question_test', auto_save=False)
app2.state.data['chief_complaint_primary'] = 'headache'
result = app2.start_intake(auto_save=False)

print(f'  First question: "{result["question"]}"')
print(f'  First intent:   {result["intent"]}')
print(f'  Generator:      {type(app2.adapter).__name__}')
assert result['question'], 'No question generated'
print('\n  PASS: Gemma generated the question successfully')

=== Question generation — still using Gemma ===
  First question: "Did the headache start suddenly and become very severe right away?"
  First intent:   ask_sudden_severe_onset
  Generator:      OllamaAdapter

  PASS: Gemma generated the question successfully
